# Day 27 — SQL Project: Data Quality Checks
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Build a comprehensive SQL-based DQ check suite across all 5 datasets
2. Implement referential integrity, null analysis, and date sanity checks
3. Produce a DQ scorecard DataFrame summarising all check results

---
> **SAP Context:** Before loading data into SAP BW (via transformation rules or DSO activation), SAP runs similar checks: mandatory field validation (InfoObject with required flag), referential integrity (SID checks against master data), and date consistency checks in the transformation. This notebook builds the equivalent of a BW DQ monitor in Python/SQL.


## 0. Setup

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('../../data/raw')

# In-memory DuckDB — register CSVs as tables/views
con = duckdb.connect()

tables = {
    'materials':    'materials_inventory.csv',
    'sales_orders': 'sales_orders.csv',
    'cost_centers': 'cost_center_actuals.csv',
    'hr':           'hr_headcount.csv',
    'bw_kpis':      'bw_sales_kpis.csv',
}

dfs = {}
for tname, fname in tables.items():
    df = pd.read_csv(DATA / fname)
    con.register(tname, df)
    dfs[tname] = df

def q(sql):
    return con.sql(sql).df()

# DQ scorecard accumulator
dq_results = []

def add_check(name, status, total, failures, notes=''):
    """Append a DQ check result to the scorecard."""
    pct = round(failures / total * 100, 2) if total > 0 else 0.0
    dq_results.append({
        'check_name': name,
        'status': status,
        'record_count': total,
        'failure_count': failures,
        'failure_pct': pct,
        'notes': notes
    })

print('Setup complete.')


## 1. Orphaned Records — Sales Orders with MATNRs Not in Materials Master

In [ ]:
# YOUR SOLUTION
sql_orphan = '''
SELECT COUNT(*) AS total,
       SUM(CASE WHEN m.MATNR IS NULL THEN 1 ELSE 0 END) AS orphaned
FROM sales_orders s
LEFT JOIN materials m ON s.MATNR = m.MATNR
'''

result = q(sql_orphan).iloc[0]
total, failures = int(result['total']), int(result['orphaned'])
status = 'PASS' if failures == 0 else ('WARN' if failures / total < 0.05 else 'FAIL')
add_check('Orphaned MATNR in Sales Orders', status, total, failures,
          'Sales order lines with no matching material master record')

print(f'Orphaned MATNRs: {failures} / {total}  → {status}')

# Show the orphaned records
q('''
SELECT s.VBELN, s.POSNR, s.MATNR, s.ERDAT
FROM sales_orders s
LEFT JOIN materials m ON s.MATNR = m.MATNR
WHERE m.MATNR IS NULL
LIMIT 10
''')

## 2. Duplicate Primary Keys

In [ ]:
# YOUR SOLUTION
# Sales orders: PK = VBELN + POSNR
dup_sql = '''
SELECT COUNT(*) AS total_rows,
       COUNT(*) - COUNT(DISTINCT VBELN || '-' || CAST(POSNR AS TEXT)) AS duplicate_pks
FROM sales_orders
'''
r = q(dup_sql).iloc[0]
total, failures = int(r['total_rows']), int(r['duplicate_pks'])
status = 'PASS' if failures == 0 else 'FAIL'
add_check('Duplicate PKs: sales_orders (VBELN+POSNR)', status, total, failures)
print(f'Sales order duplicates: {failures} / {total} → {status}')

# HR: PK = PERNR
r2 = q('SELECT COUNT(*) AS t, COUNT(*) - COUNT(DISTINCT PERNR) AS d FROM hr').iloc[0]
status2 = 'PASS' if int(r2['d']) == 0 else 'FAIL'
add_check('Duplicate PKs: hr (PERNR)', status2, int(r2['t']), int(r2['d']))
print(f'HR duplicates: {int(r2["d"])} / {int(r2["t"])} → {status2}')

# Materials: PK = MATNR + WERKS + LGORT
r3 = q("SELECT COUNT(*) AS t, COUNT(*) - COUNT(DISTINCT MATNR||WERKS||LGORT) AS d FROM materials").iloc[0]
status3 = 'PASS' if int(r3['d']) == 0 else 'FAIL'
add_check('Duplicate PKs: materials (MATNR+WERKS+LGORT)', status3, int(r3['t']), int(r3['d']))
print(f'Materials duplicates: {int(r3["d"])} / {int(r3["t"])} → {status3}')

## 3. Null Analysis on Required Fields

In [ ]:
# YOUR SOLUTION
null_checks = [
    ('sales_orders', 'VBELN',   'Sales order number cannot be null'),
    ('sales_orders', 'MATNR',   'Material number cannot be null'),
    ('sales_orders', 'NETWR',   'Net value cannot be null'),
    ('sales_orders', 'ERDAT',   'Creation date cannot be null'),
    ('materials',    'MATNR',   'Material number cannot be null'),
    ('materials',    'STPRS',   'Standard price cannot be null'),
    ('hr',           'PERNR',   'Personnel number cannot be null'),
    ('hr',           'HIRE_DATE','Hire date cannot be null'),
    ('hr',           'SALARY',  'Salary cannot be null'),
    ('cost_centers', 'KOSTL',   'Cost center cannot be null'),
    ('cost_centers', 'ACTUAL_AMT','Actual amount cannot be null'),
]

for table, col, note in null_checks:
    r = q(f'SELECT COUNT(*) AS t, SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS n FROM {table}').iloc[0]
    t, n = int(r['t']), int(r['n'])
    status = 'PASS' if n == 0 else ('WARN' if n/t < 0.01 else 'FAIL')
    add_check(f'NULL check: {table}.{col}', status, t, n, note)
    if n > 0:
        print(f'WARN: {table}.{col} has {n} nulls')

print('Null checks complete.')

## 4. Referential Integrity — Cost Center in HR Must Exist in Cost Center Actuals

In [ ]:
# YOUR SOLUTION
sql_ri = '''
SELECT COUNT(DISTINCT h.KOSTL) AS total_hr_costcenters,
       SUM(CASE WHEN c.KOSTL IS NULL THEN 1 ELSE 0 END) AS orphaned_kostl
FROM (SELECT DISTINCT KOSTL FROM hr WHERE KOSTL IS NOT NULL) h
LEFT JOIN (SELECT DISTINCT KOSTL FROM cost_centers) c ON h.KOSTL = c.KOSTL
'''
r = q(sql_ri).iloc[0]
t, n = int(r['total_hr_costcenters']), int(r['orphaned_kostl'])
status = 'PASS' if n == 0 else ('WARN' if n/t < 0.1 else 'FAIL')
add_check('RI: HR.KOSTL exists in cost_centers', status, t, n,
          'HR employees assigned to cost centers not in CO master data')
print(f'Orphaned KOSTL in HR: {n} / {t} → {status}')

# Show orphaned cost centers
q('''
SELECT DISTINCT h.KOSTL
FROM hr h
LEFT JOIN (SELECT DISTINCT KOSTL FROM cost_centers) c ON h.KOSTL = c.KOSTL
WHERE c.KOSTL IS NULL AND h.KOSTL IS NOT NULL
LIMIT 10
''')

## 5. Date Range Sanity Checks

In [ ]:
# YOUR SOLUTION
REFERENCE_DATE = '2026-04-01'

# 5a: Order dates in the future
r = q(f"SELECT COUNT(*) AS t, SUM(CASE WHEN ERDAT > '{REFERENCE_DATE}' THEN 1 ELSE 0 END) AS n FROM sales_orders").iloc[0]
status = 'PASS' if int(r['n']) == 0 else 'FAIL'
add_check('Date sanity: Sales order ERDAT not in future', status, int(r['t']), int(r['n']),
          f'Order creation dates after {REFERENCE_DATE}')
print(f'Future-dated orders: {r["n"]} → {status}')

# 5b: Hire date in the future
r = q(f"SELECT COUNT(*) AS t, SUM(CASE WHEN HIRE_DATE > '{REFERENCE_DATE}' THEN 1 ELSE 0 END) AS n FROM hr").iloc[0]
status = 'PASS' if int(r['n']) == 0 else 'FAIL'
add_check('Date sanity: HR HIRE_DATE not in future', status, int(r['t']), int(r['n']))
print(f'Future hire dates: {r["n"]} → {status}')

# 5c: Termination date before hire date
r = q("SELECT COUNT(*) AS t, SUM(CASE WHEN TERM_DATE IS NOT NULL AND TERM_DATE < HIRE_DATE THEN 1 ELSE 0 END) AS n FROM hr").iloc[0]
status = 'PASS' if int(r['n']) == 0 else 'FAIL'
add_check('Date sanity: TERM_DATE >= HIRE_DATE', status, int(r['t']), int(r['n']),
          'Employees where termination date precedes hire date')
print(f'TERM before HIRE: {r["n"]} → {status}')

# 5d: Last movement date in the future
r = q(f"SELECT COUNT(*) AS t, SUM(CASE WHEN LAST_MOVEMENT_DATE > '{REFERENCE_DATE}' THEN 1 ELSE 0 END) AS n FROM materials WHERE LAST_MOVEMENT_DATE IS NOT NULL").iloc[0]
status = 'PASS' if int(r['n']) == 0 else 'WARN'
add_check('Date sanity: LAST_MOVEMENT_DATE not in future', status, int(r['t']), int(r['n']))
print(f'Future movement dates: {r["n"]} → {status}')

## 6. DQ Scorecard

In [ ]:
# YOUR SOLUTION
scorecard = pd.DataFrame(dq_results)

# Color the status column
def color_status(val):
    if val == 'PASS': return 'background-color: #ccffcc; color: #090'
    if val == 'WARN': return 'background-color: #fff3cc; color: #880'
    if val == 'FAIL': return 'background-color: #ffcccc; color: #900'
    return ''

styled = (
    scorecard.style
    .applymap(color_status, subset=['status'])
    .format({'failure_pct': '{:.2f}%', 'record_count': '{:,}', 'failure_count': '{:,}'})
    .set_caption('Data Quality Scorecard')
    .hide(axis='index')
)

print(f"=== DQ Summary: {(scorecard['status']=='PASS').sum()} PASS | "
      f"{(scorecard['status']=='WARN').sum()} WARN | "
      f"{(scorecard['status']=='FAIL').sum()} FAIL ===\n")

print(scorecard[['check_name','status','record_count','failure_count','failure_pct']].to_string(index=False))
styled

## Daily Prompt
**Answer independently:**

1. Add a new DQ check: validate that all KUNNR (customer numbers) in sales orders follow the format "CUST-NNNN" (4 digits). Use SQLite's LIKE or GLOB operator.
2. In SAP BW, transformation rules can include "routine checks" (ABAP code). What are the 3 most common data quality issues SAP BW transformations catch at load time?
3. The DQ scorecard currently shows PASS/FAIL/WARN. Extend it: add a `severity` column ('Critical', 'High', 'Medium') based on the check type and failure percentage. Define your own severity logic and document it.
